# Проект: Многоцелевая модель для NER + event-CLS

Этот Jupyter-ноутбук - пошаговый шаблон для выполнения проекта по объединённой (multi-task) модели, решающей **NER (BIO, token-level)** и **CLS (document-level multihot событий/отношений)** на датасете NEREL.

Внимание: вам нужно реализовать весь рабочий код - в ноутбуке предустановлены только парсеры строкового формата. Все остальные ячейки служат как инструкции / места для вашего кода.


#### Структура ноутбука 

1. Подготовка окружения (пути, seed, imports)
2. EDA - загрузка jsonl, обзор, графики, выводы 
3. Парсинг и таргеты - здесь уже есть парсеры строкового формата NEREL; нужно реализовать сбор примеров (`build_examples_from_nerel`) 
4. Токенизация, выравнивание меток, DataLoader - реализовать `tokenize_and_align_labels`, Dataset/Collator 
5. Модель (JointModel) и кастомный loss (uncertainty-weighting) - реализовать модельный класс и loss
6. Тренировка/валидация - training loop, оптимизатор, scheduler, логирование метрик
7. Инференс и анализ ошибок - реализовать inference pipeline и примеры



In [196]:
import json
from random import seed
from typing import List
import re
from collections import Counter

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score, accuracy_score
import seaborn as sns
from matplotlib import pyplot as plt
import pandas as pd
from tqdm import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer
from transformers import DataCollatorForTokenClassification
import torch.nn.functional as F


In [ ]:
seed(42)
plt.style.use('seaborn-v0_8-bright')

##### 1. EDA

Цели:

- Прочитать первые 50–200 записей `train.jsonl` (путь `/data/train.jsonl`).
- Посчитать частоты: entity types, relation/event types.
- Построить графики: топ-15 entity types, распределение длины текстов, число сущностей на документ.
- Написать 2–3 коротких вывода в Markdown: что можно ожидать при моделировании (редкие типы, длинные документы и т. п.).




In [ ]:
data_path = 'nerel/'

with open(data_path + 'train.jsonl', 'r') as json_file:
    json_list = list(json_file)
    train_data = [json.loads(elem) for elem in json_list]
with open(data_path + 'test.jsonl', 'r') as json_file:
    json_list = list(json_file)
    test_data = [json.loads(elem) for elem in json_list]
with open(data_path + 'dev.jsonl', 'r') as json_file:
    json_list = list(json_file)
    dev_data = [json.loads(elem) for elem in json_list]

with open(data_path + 'rel_types.jsonl', 'r') as json_file:
    json_list = list(json_file)
    rel_types_data = [json.loads(elem) for elem in json_list]
with open(data_path + 'ent_types.jsonl', 'r') as json_file:
    json_list = list(json_file)
    ent_types_data = [json.loads(elem) for elem in json_list]

In [ ]:
rel_types_dict = {elem['type']: 0 for elem in rel_types_data}
ent_types_dict = {elem['type']: 0 for elem in ent_types_data}

In [ ]:
print(ent_types_dict)

In [ ]:
print(rel_types_dict)

In [ ]:
train_data[0].keys()

In [ ]:
text = train_data[0]['text']
entities = train_data[0]['entities']
relations = train_data[0]['relations']
links = train_data[0]['links']

In [ ]:
print(text[:100])

In [ ]:
entities[:5]

In [ ]:
entities[5-1], entities[70-1]

In [ ]:
relations[:10]

In [ ]:
ent_counts, rel_counts, char_counts = [], [], []
for elem in train_data:
    text = elem['text']
    entities = elem['entities']
    relations = elem['relations']
    links = elem['links']
    ent_counts.append(len(entities))
    rel_counts.append(len(relations))
    char_counts.append(len(text))

    for ent in entities:
        ent_types_dict[ent.split('\t')[1].split()[0]] += 1
    for rel in relations:
        rel_types_dict[rel.split('\t')[1].split()[0]] += 1

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 2))
for i, cnt, title_suffix in zip(range(3), 
             [char_counts, ent_counts, rel_counts], 
             ['символов', 'entities', 'ralations']):
    ax[i].hist(cnt, bins=30)
    ax[i].set_title(f'Распределение кол-ва\n{title_suffix} в текстах')
    ax[i].tick_params(axis='x', labelrotation=45)

In [ ]:
df = pd.DataFrame({'text_size': char_counts, 'entity_count': ent_counts})
plt.figure(figsize=(5, 3))
fig = sns.scatterplot(x=df['text_size'], y=df['entity_count'])
fig.tick_params(axis='x', rotation=90)
plt.title('Количество entity к длине текста');

In [ ]:
df = pd.DataFrame(list(ent_types_dict.items()), columns = ['entity', 'count'])
df = df.sort_values('count', ascending=False).reset_index(drop=True)
plt.figure(figsize=(7, 2))
fig = sns.barplot(x=df['entity'], y=df['count'])
fig.tick_params(axis='x', rotation=90)
plt.title('Количество различных ENTITY в TRAIN-сете');

In [ ]:
df = pd.DataFrame(list(rel_types_dict.items()), columns = ['relations', 'count'])
df = df.sort_values('count', ascending=False).reset_index(drop=True)
plt.figure(figsize=(9, 2))
fig = sns.barplot(x=df['relations'], y=df['count'])
fig.tick_params(axis='x', rotation=90)
plt.title('Количество различных RELATION в TRAIN-сете');

In [ ]:
EVENT_LIST = list(rel_types_dict.keys())

In [ ]:
EVENT_LIST

##### Парсинг и подготовка таргетов

Ниже - две заранее подготовленные функции парсинга строкового формата NEREL. Их вы получаете и можете использовать сразу (не меняйте имена).

Ожидаемые дополнительные функции, которые вы должны реализовать:

- `build_examples_from_nerel(records: List[dict], event_list: List[str]) -> List[dict]` - для каждого документа вернуть словарь с полями: `text`, `tokens` (word-tokenization по пробелам), `token_spans` (символьные оффсеты слов), `tags` (BIO per token), `cls_vec` (multihot длиной len(event_list)).

- `make_event_list(records, K=30)` - собрать топ-K типов событий/relations и вернуть список.



**Правила BIO и сопоставления:**

- Токенизация для BIO - простая `text.split()` (по пробелам). Офсеты токенов вычисляются на основе поиска токена в тексте (учтите повторы; используйте скользящий указатель).
- Для каждой сущности (start, end - символьные оффсеты) пометьте токены, которые пересекаются с интервалом сущности.
- Метки: `B-TYPE`, `I-TYPE`, `O`.



In [ ]:


# Функции парсинга строкового формата NEREL
def parse_entity_line(line: str):
    parts = line.split('\t')
    if len(parts) < 3:
        return None
    ent_id = parts[0].strip()
    type_pos = parts[1].strip()
    text = parts[2].strip() if len(parts) > 2 else ''
    m = re.match(r'(\S+)\s+(\d+)\s+(\d+)', type_pos)
    if not m:
        return None
    ent_type = m.group(1)
    start = int(m.group(2))
    end = int(m.group(3))
    return {'id': ent_id, 'type': ent_type, 'start': start, 'end': end, 'text': text}

def parse_relation_line(line: str):
    parts = line.split('\t')
    if len(parts) < 2:
        return None
    rel_id = parts[0].strip()
    body = parts[1].strip()
    m = re.match(r'(\S+)\s+Arg1:(\S+)\s+Arg2:(\S+)', body)
    if not m:
        return None
    rel_type = m.group(1)
    arg1 = m.group(2); arg2 = m.group(3)
    return {'id': rel_id, 'type': rel_type, 'arg1': arg1, 'arg2': arg2}

In [ ]:
def whitespace_tokenize_with_offsets(text: str):
    tokens = []
    spans = []
    for m in re.finditer(r'\S+', text):
        tokens.append(m.group())
        spans.append((m.start(), m.end()))
    return tokens, spans 

In [ ]:
def make_event_list(records, K=30):
    relations_dict = {}

    for rec in records:
        relations = list(set([parse_relation_line(line)['type'] for line in rec['relations']]))

        for rel in relations:
            if rel in relations_dict:
                relations_dict[rel] += 1
            else:
                relations_dict[rel] = 1
    data = list(relations_dict.items())
    data = sorted(data, key=lambda x: x[1], reverse=True)

    return [d[0] for d in data[:K]]

In [ ]:
def build_examples_from_nerel(records: List[dict], event_list: List[str]) -> List[dict]:

    mlb = MultiLabelBinarizer()
    mlb.fit([event_list])

    examples = []
    
    for rec in records:
        tokens, token_spans = whitespace_tokenize_with_offsets(rec['text'])
        token_labels = ["O"] * len(tokens)

        # Пройдите по rec.objects и заполните token_labels
        # Ваш код здесь
        for line in rec['entities']:
            obj = parse_entity_line(line)
            base_type = obj['type']
            span_start = obj['start']
            span_end = obj['end']


            overlapping_idxs = []
            for i, (t_start, t_end) in enumerate(token_spans):
                if not (t_end <= span_start or t_start >= span_end):
                    overlapping_idxs.append(i)
            if not overlapping_idxs:
                # можно логировать: print(f"No overlap for span {span_start}-{span_end} in doc {rec.id}")
                continue
            for j, tok_idx in enumerate(overlapping_idxs):
                if token_labels[tok_idx] != "O":
                    continue
                prefix = "B" if j == 0 else "I"
                token_labels[tok_idx] = f"{prefix}-{base_type}"


        relations = set([parse_relation_line(line)['type'] for line in rec['relations']])
        cls_vec = mlb.transform([relations])[0]

        examples.append({
            "text": rec['text'],
            "tokens": tokens,
            "tags": token_labels,
            "cls_vec": cls_vec,
        })

    return examples

In [ ]:
EVENT_LIST = make_event_list(train_data, K=30)
len(EVENT_LIST)

In [ ]:
train_examples = build_examples_from_nerel(train_data, EVENT_LIST)
test_examples = build_examples_from_nerel(test_data, EVENT_LIST)
dev_examples = build_examples_from_nerel(dev_data, EVENT_LIST)

In [ ]:
train_examples[0].keys()

In [ ]:
train_examples[0]['tags'][:5]

In [ ]:
train_examples[0]['cls_vec']

In [ ]:
# for examples in [train_examples, test_examples]:
#     for ex in examples:
#         ex["tags"] = [label2id[t] for t in ex["tags"]]

In [ ]:
# dataset = DatasetDict({"train": train_examples, "test": test_examples})

##### 3. Токенизация и выравнивание меток

Задачи:

- Выбрать `AutoTokenizer(..., use_fast=True)`.
- Реализовать `tokenize_and_align_labels(examples, tokenizer, label2id, max_length)`:
  - Токенизировать текст (return_offsets_mapping
  - Преобразовать word-level BIO метки в token-level метки (subword → label = -100 / ignore_index, для первых субтокенов ставится соответствующий тег `B-`/`I-`)
  - Вернуть словарь с `input_ids`, `attention_mask`, `labels` (token-level), `cls_labels`

- Собрать `torch.utils.data.Dataset` и `DataLoader`. Можно использовать `DataCollatorForTokenClassification` либо сделать кастомный collator, который возвращает батчи с `cls_labels`.



In [ ]:
unique_labels = set()
for ex in train_examples+test_examples:
    unique_labels.update(ex["tags"])
unique_labels.add("O")
label_list = sorted(unique_labels)
label2id = {lab: i for i, lab in enumerate(label_list)}
id2label = {i: lab for lab, i in label2id.items()}

In [ ]:
len(unique_labels)

In [ ]:
MODEL_NAME = "cointegrated/rubert-tiny2"
# attention_model = AutoModel.from_pretrained(model_name, output_attentions=True) # Ваш код здесь
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME) # Ваш код здесь

In [ ]:
# Для подбора max_length
tokenized = tokenizer(
    [e['tokens'] for e in train_examples],
    is_split_into_words=True,
    truncation=True,
    padding="max_length",
    max_length=1000,
)
tkn_lengths = [sum([el == 1 for el in atm])  for atm in  tokenized['attention_mask']]
sns.displot(tkn_lengths, height=2, aspect=1.5);

In [ ]:
MAX_LENGTH = 512

In [ ]:
train_examples[0].keys()

In [ ]:
tokenized = tokenizer(
    [e['tokens'] for e in train_examples[:2]],
    is_split_into_words=True,
    truncation=True,
    padding="max_length",
    max_length=MAX_LENGTH,
)

In [ ]:
tokenized

In [ ]:
label2id

In [ ]:
def tokenize_and_align_labels(examples, tokenizer, label2id, max_length):

    tokenized = tokenizer(
        [e['tokens'] for e in examples],
        is_split_into_words=True,
        truncation=True,
        padding="max_length",
        max_length=max_length,
    )
    labels = []
    cls_labels = []

    for i, exam in enumerate(examples):
        cls_labels.append(exam['cls_vec'].tolist())
        word_labels = [label2id[t] for t in exam['tags']]

        word_ids = tokenized.word_ids(batch_index=i)
        label_ids = []
        prev_word_idx = None
        for word_idx in word_ids:
            # Условие: если word_idx == None, то это padding/special token
            # Иначе, если word_idx != prev_word_idx, то это начало нового слова
            # Ваш код здесь
            if word_idx is None:
                label_ids.append(-100)  # .append(-100)
            elif word_idx != prev_word_idx:
                label_ids.append(word_labels[word_idx])
            else:
                label_ids.append(-100)  # .append(-100)
            prev_word_idx = word_idx
        labels.append(label_ids)
    
    return {
        'input_ids': tokenized['input_ids'],
        'attention_mask': tokenized['attention_mask'], 
        'labels': labels, 
        'cls_labels': cls_labels
    }

In [ ]:
train_tokenized = tokenize_and_align_labels(train_examples, tokenizer, label2id, max_length=MAX_LENGTH)
test_tokenized = tokenize_and_align_labels(test_examples, tokenizer, label2id, max_length=MAX_LENGTH)

In [ ]:
class CustomDictionaryDataset(Dataset):
    """
    A custom PyTorch Dataset class to load data from a dictionary.
    The dictionary is expected to have the same number of items for each key.
    """
    def __init__(self, data_dict):
        # super().__init__()
        self.data_dict = data_dict
        # Assuming all lists in the dictionary have the same length
        self.length = len(next(iter(data_dict.values())))

    def __len__(self):
        # Return the total number of samples in the dataset
        return self.length

    def __getitem__(self, idx):
        # Return a single sample as a dictionary at the given index
        sample = {key: torch.tensor(self.data_dict[key][idx]) for key in self.data_dict}
        return sample

In [ ]:
type(train_tokenized)

In [ ]:
train_dataset = CustomDictionaryDataset(train_tokenized)
test_dataset = CustomDictionaryDataset(test_tokenized)

In [ ]:
len(label2id)

In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer)

train_dataloader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=data_collator
)

test_dataloader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=data_collator
)

In [ ]:
for batch in test_dataloader:
    break

##### 4. Модель: `JointModel` + custom loss (uncertainty weighting)


In [ ]:
import torch.nn as nn
import timm

In [ ]:
class Config:
    SEED = 42

    TEXT_MODEL_NAME = MODEL_NAME
    # IMAGE_MODEL_NAME = "tf_efficientnet_b0"

    # TEXT_MODEL_UNFREEZE = "encoder.layer.11|pooler"
    # IMAGE_MODEL_UNFREEZE = "blocks.6|conv_head|bn2"

    BATCH_SIZE = 16 #256
    
    TEXT_LR = 1e-3
    NER_LR = 1e-3
    MULTIHOT_LR = 1e-3
    LOSS_LR = 1e-3

    EPOCHS = 2 #30
    DROPOUT = 0.2
    HIDDEN_DIM = 256
    MAX_LENGTH = MAX_LENGTH

    NER_LABELS_COUNT = len(label2id)

    MULTIHOT_SIZE = len(EVENT_LIST)

    TEXT_MODEL_UNFREEZE = "encoder.layer.2.output|pooler"

    SAVE_PATH = "models/best_model.pth"


In [ ]:
class JointModel(nn.Module):
    def __init__(self, config,):
        super().__init__()
        self.text_model = AutoModel.from_pretrained(config.TEXT_MODEL_NAME)

        self.ner_layer = nn.Sequential(
            nn.Linear(
                in_features=(
                    self.text_model.config.hidden_size
                ),
                out_features=config.HIDDEN_DIM,
                bias=True
            ),
            nn.LayerNorm(config.HIDDEN_DIM),
            nn.Hardswish(),
            nn.Dropout(p=config.DROPOUT),
            nn.Linear(in_features=config.HIDDEN_DIM,
                        out_features=config.NER_LABELS_COUNT, bias=True),
            nn.Softmax(dim=1)
        )

        self.multihot_layer = nn.Sequential(
            nn.Linear(
                in_features=(
                    self.text_model.config.hidden_size
                ),
                out_features=config.HIDDEN_DIM,
                bias=True
            ),
            nn.LayerNorm(config.HIDDEN_DIM),
            nn.Hardswish(),
            nn.Dropout(p=config.DROPOUT),
            nn.Linear(in_features=config.HIDDEN_DIM,
                        out_features=config.MULTIHOT_SIZE, bias=True),
            nn.Softmax(dim=1)
        )

    def forward(self, input_ids, attention_mask):
        text_model_output = self.text_model(
            input_ids=input_ids,            
            attention_mask=attention_mask,
        )
        
        token_features = text_model_output.last_hidden_state
        cls_features = text_model_output.last_hidden_state[:, -1, :]  # pooler_output 
        
        ner_output = self.ner_layer(token_features)


        # multihot_output = self.multihot_layer(text_model_output.pooler_output)
        multihot_output = self.multihot_layer(cls_features)

        return ner_output, multihot_output

In [ ]:
class KendallLossWeighting(nn.Module):
    """
    Implements the uncertainty-based loss weighting from Kendall et al. 2018
    for multi-task learning.
    """
    def __init__(self, num_tasks):
        super(KendallLossWeighting, self).__init__()
        # Initialize log(sigma^2) for each task as a learnable parameter
        # using zeros as a starting point.
        self.log_sigma_sq = nn.Parameter(torch.zeros(num_tasks))

    def forward(self, losses):
        """
        Calculates the weighted loss sum.

        Args:
            losses (list of Tensors): List of individual task losses.
        
        Returns:
            Tensor: The combined, uncertainty-weighted loss.
        """
        assert len(losses) == len(self.log_sigma_sq), "Number of losses must match number of tasks"

        total_loss = 0
        for i, loss in enumerate(losses):
            # Formula from the paper: L_weighted = (1 / (2 * sigma_i^2)) * L_i + log(sigma_i)
            # We work with log_sigma_sq for numerical stability.
            
            # GOOGLE AI
            # # precision = 1 / (2 * sigma_i^2) = (1/2) * exp(-log_sigma_sq)
            # precision = 0.5 * torch.exp(-self.log_sigma_sq[i])
            # # log(sigma_i) = 0.5 * log(sigma_i^2) = 0.5 * log_sigma_sq
            # log_sigma = 0.5 * self.log_sigma_sq[i]  

            # PRACTICUM
            precision = torch.exp(-2.0 * self.log_sigma_sq[i])
            log_sigma = self.log_sigma_sq[i]

            total_loss += precision * loss + log_sigma
            
        return total_loss

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
device

In [ ]:
config = Config()
model = JointModel(config)
model.to(device)

loss_ner = nn.CrossEntropyLoss(ignore_index=-100)
loss_cls = nn.BCEWithLogitsLoss()

num_tasks = 2
loss_weighter = KendallLossWeighting(num_tasks)

In [ ]:
for param, _ in model.named_parameters():
    print(param)

In [ ]:
def set_requires_grad(module, unfreeze_pattern="", verbose=False):
    if len(unfreeze_pattern) == 0:
        for param, _ in module.named_parameters():
            param.requires_grad = False
        return

    pattern = re.compile(unfreeze_pattern)

    for name, param in module.named_parameters():
        if pattern.search(name):
            param.requires_grad = True
            if verbose:
                print(f"Разморожен слой: {name}")
        else:
            param.requires_grad = False

In [ ]:
# for i, (name, param) in enumerate(model.text_model.named_parameters()):
#     print(i, name, param.shape)

In [ ]:
set_requires_grad(model,
                  unfreeze_pattern="encoder.layer.2|ner_layer|multihot_layer",
                  # unfreeze_pattern="ner_layer|multihot_layer",
                  verbose=True)

In [ ]:
optimizer = torch.optim.Adam([
    {'params': model.text_model.parameters(), 'lr': config.TEXT_LR},
    {'params': model.ner_layer.parameters(), 'lr': config.NER_LR},
    {'params': model.multihot_layer.parameters(), 'lr': config.MULTIHOT_LR},
    {'params': loss_weighter.parameters(), 'weight_decay': 0, 'lr': config.LOSS_LR} # Often no weight decay on sigma
])

In [228]:
treshold_cls = 0.5
with torch.no_grad():
    ner_preds_list, ner_true_list = [], []
    cls_preds_list, cls_true_list = [], []
    for batch in tqdm(test_dataloader):

        batch = {k: v.to(device) for k, v in batch.items()}
        ner_output, multihot_output = model(batch['input_ids'],                                            
                                            batch['attention_mask'],)           
        loss_ner_value = loss_ner(ner_output.view(-1, config.NER_LABELS_COUNT), batch['labels'].view(-1))
        loss_cls_value = loss_cls(multihot_output, batch['cls_labels'].float())

        total_weighted_loss = loss_weighter(losses=[loss_ner_value, loss_cls_value])

        preds = torch.argmax(ner_output, dim=-1).cpu().tolist()  # list длины seq_len
        preds_cls = (multihot_output > treshold_cls).int()
        break

  0%|          | 0/47 [00:00<?, ?it/s]


In [201]:
ner_output.shape

torch.Size([16, 512, 57])

##### 5. Training / Validation



In [230]:
# Функция, выравнивающая предсказания модели и реальные метки (на уровне tokenized_dataset)
def get_flat_labels_and_preds_from_model(tokenized_split, model, device, max_samples=None, treshold_cls=0.5):
    """
    tokenized_split: dataset split (list-like of examples with keys 'input_ids','attention_mask','labels')
    Возвращает flat lists: y_true (ints), y_pred (ints)
    """
    y_true = []
    y_pred = []

    y_true_cls = []
    y_pred_cls = []
    for i, ex in enumerate(tokenized_split):
        if max_samples is not None and i >= max_samples:
            break

        input_ids = ex["input_ids"].unsqueeze(0).to(torch.long).to(device)
        attention_mask = ex["attention_mask"].unsqueeze(0).to(torch.long).to(device)

        with torch.no_grad():
            # outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            # logits = outputs.logits  # shape (1, seq_len, num_labels)
            logits, multihot = model(input_ids, attention_mask)

            preds = torch.argmax(logits, dim=-1).squeeze(0).cpu().tolist()  # list длины seq_len

            preds_cls = (multihot.squeeze(0) > treshold_cls).int()

        # Истинные метки (включая -100 для пэддинга/ignored)
        true_labels = ex["labels"]  # список длиной seq_len; элементы -100 или id
        true_labels_cls = ex["cls_labels"]

        # Фильтруем позиции, где true != -100
        filtered_true = []
        filtered_pred = []
        for p, t in zip(preds, true_labels):
            if t == -100:
                continue
            filtered_true.append(int(t))
            filtered_pred.append(int(p))

        # Обрежем на минимальную длину (на случай рассинхронизации)
        minlen = min(len(filtered_true), len(filtered_pred))
        if minlen == 0:
            continue
        y_true.extend(filtered_true[:minlen])
        y_pred.extend(filtered_pred[:minlen])

        y_true_cls.extend(preds_cls.tolist())
        y_pred_cls.extend(true_labels_cls.tolist())


    return y_true, y_pred, y_true_cls, y_pred_cls

In [231]:
def validate(model, val_loader, device, loss_weighter): 
    model.eval()
    n_batches = 0
    total_loss = 0
    total_loss_ner = 0
    total_loss_cls = 0
    loss_ner = nn.CrossEntropyLoss(ignore_index=-100)
    loss_cls = nn.BCEWithLogitsLoss()
    with torch.no_grad():
        for batch in tqdm(val_loader):

            batch = {k: v.to(device) for k, v in batch.items()}
            ner_output, multihot_output = model(batch['input_ids'],                                            
                                                batch['attention_mask'],)           
            loss_ner_value = loss_ner(ner_output.view(-1, config.NER_LABELS_COUNT), batch['labels'].view(-1))
            loss_cls_value = loss_cls(multihot_output, batch['cls_labels'].float())

            total_weighted_loss = loss_weighter(losses=[loss_ner_value, loss_cls_value])
            total_loss += total_weighted_loss.item()
            total_loss_ner += loss_ner_value
            total_loss_cls += loss_cls_value
            n_batches += 1
        
        avg_loss = total_loss / n_batches if n_batches > 0 else 0.0
        avg_loss_ner = total_loss_ner / n_batches if n_batches > 0 else 0.0
        avg_loss_cls = total_loss_cls / n_batches if n_batches > 0 else 0.0

    return avg_loss, avg_loss_ner, avg_loss_cls 

In [ ]:
# 3) Loop обучения
val_loss, val_loss_ner, val_loss_cls = validate(model, test_dataloader, device, loss_weighter)
print(f"INIT VALIDATION: avg weighted loss: {val_loss:.4f} / avg loss_ner: {val_loss_ner:.4f} / val_loss_cls: {val_loss_cls:.4f}\n")

for epoch in range(config.EPOCHS):
    model.train()
    total_loss = 0.0
    n_batches = 0

    total_loss_ner = 0
    total_loss_cls = 0

    for batch in tqdm(train_dataloader, desc=f"Epoch {epoch+1}"):
    # for batch in train_dataloader:    
        optimizer.zero_grad()

        batch = {k: v.to(device) for k, v in batch.items()}
        ner_output, multihot_output = model(batch['input_ids'],                                            
                                            batch['attention_mask'],)
        
        loss_ner_value = loss_ner(ner_output.view(-1, config.NER_LABELS_COUNT), batch['labels'].view(-1))
        loss_cls_value = loss_cls(multihot_output, batch['cls_labels'].float())


        total_weighted_loss = loss_weighter(losses=[loss_ner_value, loss_cls_value])
        total_weighted_loss.backward()
        optimizer.step()

        total_loss += total_weighted_loss.item()
        total_loss_ner += loss_ner_value
        total_loss_cls += loss_cls_value
        n_batches += 1

    train_loss = total_loss / n_batches if n_batches > 0 else 0.0
    train_loss_ner = total_loss_ner / n_batches if n_batches > 0 else 0.0
    train_loss_cls = total_loss_cls / n_batches if n_batches > 0 else 0.0

    

    # Valiation
    val_loss, val_loss_ner, val_loss_cls  = validate(model, test_dataloader, device, loss_weighter)

    y_true_ner_val, y_pred_ner_val, y_true_cls_val, y_pred_cls_val = get_flat_labels_and_preds_from_model(
        test_dataset, 
        model, 
        device, 
        max_samples=None
    )
    f1_ner_macro = f1_score(y_true_ner_val, y_pred_ner_val)
    f1_ner_macro = f1_score(y_true_ner_val, y_pred_ner_val)


    print(f"TRAIN: avg weighted loss: {train_loss:.4f} / avg loss_ner: {train_loss_ner:.4f} / avg_loss_cls: {train_loss_cls:.4f}\n"
          f"VALID: avg weighted loss: {val_loss:.4f} / avg loss_ner: {val_loss_ner:.4f} / val_loss_cls: {val_loss_cls:.4f}\n")
    


In [ ]:
# torch.save(model.state_dict(), config.SAVE_PATH)

In [ ]:
config.SAVE_PATH

In [ ]:
train_dataset

In [ ]:
batch['input_ids'].shape

In [ ]:
batch['attention_mask'].shape

In [ ]:
with torch.no_grad():
    for batch in tqdm(test_dataloader):

        batch = {k: v.to(device) for k, v in batch.items()}
        ner_output, multihot_output = model(batch['input_ids'],                                            
                                            batch['attention_mask'],)
        break

In [ ]:
batch['input_ids'].shape

In [ ]:
batch['attention_mask'].shape

In [ ]:
ner_output.shape

In [ ]:
multihot_output.shape

In [ ]:
ex["input_ids"].shape

In [ ]:
ex["input_ids"].unsqueeze(0).shape

In [ ]:
for i, ex in enumerate(test_dataset):

    with torch.no_grad():
        # outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        # logits = outputs.logits  # shape (1, seq_len, num_labels)
        logits, _ = model(ex["input_ids"].unsqueeze(0).to(device), ex["attention_mask"].unsqueeze(0).to(device))

        preds = torch.argmax(logits, dim=-1).squeeze(0).cpu().tolist()  # list длины seq_len
        break

In [ ]:
logits.shape

In [ ]:
preds

In [190]:
y_true, y_pred, y_true_cls, y_pred_cls = get_flat_labels_and_preds_from_model(test_dataset, model, device, max_samples=None)

In [191]:
print(classification_report(y_true, y_pred, target_names=list(id2label.values())))

                     precision    recall  f1-score   support

              B-AGE       0.72      0.76      0.74       138
            B-AWARD       0.00      0.00      0.00        94
             B-CITY       0.42      0.61      0.50       210
          B-COUNTRY       0.48      0.57      0.52       332
            B-CRIME       0.00      0.00      0.00        30
             B-DATE       0.79      0.70      0.74       479
          B-DISEASE       0.00      0.00      0.00        49
         B-DISTRICT       0.00      0.00      0.00        20
            B-EVENT       0.44      0.34      0.38       600
         B-FACILITY       0.02      0.03      0.03        60
           B-FAMILY       0.00      0.00      0.00        11
         B-IDEOLOGY       0.00      0.00      0.00        22
         B-LANGUAGE       0.00      0.00      0.00         8
              B-LAW       0.00      0.00      0.00        34
         B-LOCATION       0.00      0.00      0.00        38
            B-MONEY    

/home/russele7/practicum/dle/practicum_dle_sprint_5/venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/russele7/practicum/dle/practicum_dle_sprint_5/venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/russele7/practicum/dle/practicum_dle_sprint_5/venv/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control

In [198]:
f1_micro_cls = f1_score(y_true_cls, y_pred_cls, average='micro')
accuracy_cls = accuracy_score(y_true_cls, y_pred_cls) 
precision_cls = precision_score(y_true_cls, y_pred_cls) 
recall_cls = recall_score(y_true_cls, y_pred_cls) 
print(f'CLS metrics: accuracy {accuracy_cls:.4f} / precision {precision_cls:.4f} / recall {recall_cls:.4f} / f1_micro {f1_micro_cls:.4f} / ')

CLS metrics: accuracy 0.5903 / precision 0.0009 / recall 1.0000 / f1_micro 0.5903 / 


##### 6. Инференс, квантизация и анализ ошибок

Проведите качественный анализ на 8–10 примерах: где NER ошибается? Какие типы сущностей плохо определяются? Насколько квантизация может ускорить инференс и сильно ли она ухудшает модель?


##### Заключение

Этот шаблон даёт вам чёткую дорожную карту и рабочие точки, где нужно реализовать код. В ноутбуке предоставлены только парсеры строкового формата - всё остальное вы пишете самостоятельно: токенизация/выравнивание меток, датасеты, модель, loss, тренировка и анализ.

Удачи - приступайте к реализации прямо в ноутбуке!